# Progetto finale M4

Importazione librerie

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from keras.preprocessing import image
import numpy as np

Fase A: Data Pipeline e Augmentation

In [ ]:
# Cartelle con i dati
TRAINING_DIR = "rps/rps/"
TEST_DIR = "rps-test-set/rps-test-set/"

# 1. ImageDataGenerator Training (con Augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# 2. ImageDataGenerator Test
test_datagen = ImageDataGenerator(rescale=1./255)

# 3. Flusso dei dati (Generators)
train_generator = train_datagen.flow_from_directory(
    TRAINING_DIR,
    target_size=(150, 150),
    class_mode='categorical',
    batch_size=32
)

validation_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(150, 150),
    class_mode='categorical',
    batch_size=32
)

Fase B: Architettura della Rete (CNN)

In [ ]:
model = Sequential([
    # 1. Input Layer: Immagini 150x150 pixel a 3 canali (RGB)
    Input(shape=(150, 150, 3)),
    
    # 2. Blocchi Convoluzionali (Filtri che aumentano progressivamente: 32 -> 64 -> 128)
    # Primo blocco
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2, 2),
    
    # Secondo blocco
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    # Terzo blocco
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    # Quarto blocco 
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    # 3. Livelli Dense
    Flatten(),
    Dense(512, activation='relu'),
    
    # 4. Output Layer (3 neuroni per Sasso, Carta, Forbice con Softmax)
    Dense(3, activation='softmax')
])

model.summary()

Fase C: Compilazione e Training

In [ ]:
# Callback personalizzata
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if(logs.get('accuracy') is not None and logs.get('accuracy') > 0.98):
            print("\nRaggiunto il 98% di accuratezza, addestramento interrotto!")
            self.model.stop_training = True

callbacks = myCallback()

# Compilazione del modello
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Training del modello
history = model.fit(
    train_generator,
    epochs=20,
    validation_data=validation_generator,
    callbacks=[callbacks]
)

Fase D: Test e Validazione "Wild"

In [ ]:
def predict_image(image_path):
    # Carica l'immagine e la ridimensiona a 150x150
    img = image.load_img(image_path, target_size=(150, 150))
    
    # Converte in array Numpy
    x = image.img_to_array(img)
    # Aggiunge una dimensione extra (batch) -> diventa (1, 150, 150, 3)
    x = np.expand_dims(x, axis=0)
    # Normalizza i pixel (fondamentale, altrimenti le predizioni sballano!)
    x = x / 255.0
    
    # Effettua la predizione
    classes = model.predict(x, batch_size=1)
    
    # L'ordine delle classi dipende da come le ha caricate il generator (solitamente alfabetico)
    # labels: ['paper', 'rock', 'scissors']
    class_names = list(train_generator.class_indices.keys())
    predicted_class = class_names[np.argmax(classes[0])]
    
    print(f"La probabilità grezza è: {classes[0]}")
    print(f"Il modello predice che l'immagine è: {predicted_class.upper()}")


mie_foto = ['mia_sasso.jpg', 'mia_carta.jpg', 'mia_forbice.jpg']

for foto in mie_foto:
    print(f"\n--- Analizzando: {foto} ---")
    try:
        predict_image(foto)
    except FileNotFoundError:
        print(f"Errore: Non riesco a trovare il file '{foto}'. Assicurati di averlo caricato!")

Analisi del Domain Gap

Il modello ha raggiunto un'accuratezza eccellente sul Test Set originale ma con le foto reali vengono introdotte una serie di complessità visive (Sfondo, Contrasto, Illuminazione, Ombre, Texture e Rumore) che non sono presenti nei dati di addestramento.
Durante il test pratico il modello ha classificato correttamente Sasso e Carta ma ha fallito su Forbice scambiandolo per Sasso:

```text
--- Analizzando: mia_forbice.jpg ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
La probabilità grezza è: [0.13423526 0.61721605 0.24854864]
Il modello predice che l'immagine è: ROCK
```